# PRF Analysis

Visualize and summarize GaussianPRF2D fits from GLMsingle single-trial betas.

Pipeline:
1. `fit_prf.py` → volumetric parameter maps (T1w space)
2. `sample_prf_to_surface.py` → fsnative `.func.gii` files
3. `register_retinotopy.py` → neuropythy Benson atlas (`inferred_varea.mgz`)
4. This notebook → analysis per ROI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import nibabel as nib
from nilearn import image, plotting
from nilearn.maskers import NiftiMasker
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/Users/gdehol/git/value_capture')
from value_capture.utils.data import Subject, BIDS_FOLDER

BIDS = BIDS_FOLDER
PRF_DERIV = 'prf_glmsingle'
SUBJECTS = ['01', '02']

ROI_ORDER  = ['V1', 'V2', 'V3', 'hV4', 'LO1', 'LO2', 'VO1', 'VO2']
ROI_COLORS = plt.cm.tab10(np.linspace(0, 0.8, len(ROI_ORDER)))

## 1. GM mask at functional resolution

The GM probseg is resampled from T1w (~0.8 mm) to functional resolution (~1.5 mm) with **linear** interpolation.  
At 0.5 threshold, sulcal voxels with partial-volume mixing are excluded.  
At 0.1 (new default) essentially all cortex is kept; WM/CSF is still excluded.

In [ ]:
from pathlib import Path

def get_gm_prob_func(subject, session=1, threshold=None):
    """Load GM probseg resampled to functional resolution."""
    sub = Subject(subject, bids_folder=BIDS)
    anat_dir = Path(BIDS) / 'derivatives' / 'fmriprep' / f'sub-{sub.subject_id}' / f'ses-{session}' / 'anat'
    candidates = sorted(anat_dir.glob(f'sub-{sub.subject_id}*label-GM_probseg.nii.gz'))
    candidates = [c for c in candidates if 'MNI' not in c.name]
    gm_img = image.load_img(str(candidates[0]))
    ref_img = sub.get_brain_mask(session)
    gm_res = image.resample_to_img(gm_img, ref_img, interpolation='linear')
    if threshold is not None:
        return image.math_img(f'img >= {threshold}', img=gm_res)
    return gm_res

sub = Subject('01', bids_folder=BIDS)
t1w = sub.get_t1w()

gm_prob  = get_gm_prob_func('01')
gm_50    = get_gm_prob_func('01', threshold=0.5)
gm_10    = get_gm_prob_func('01', threshold=0.1)
brain_mask = sub.get_brain_mask(1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
slices = dict(cut_coords=[-10, 0, 10], display_mode='z')

plotting.plot_stat_map(gm_prob, bg_img=t1w, axes=axes[0],
                       vmin=0, vmax=1, cmap='hot', colorbar=True,
                       title='GM prob (continuous, func res)', **slices)
plotting.plot_roi(gm_50, bg_img=t1w, axes=axes[1],
                  title='GM > 0.5 (old default)', **slices)
plotting.plot_roi(gm_10, bg_img=t1w, axes=axes[2],
                  title='GM > 0.1 (new default)', **slices)
plt.tight_layout()
plt.show()

d50 = int(gm_50.get_fdata().sum())
d10 = int(gm_10.get_fdata().sum())
bm  = int(brain_mask.get_fdata().sum())
print(f'Brain mask: {bm:,} voxels')
print(f'GM > 0.5:   {d50:,} voxels  ({100*d50/bm:.0f}% of brain mask)')
print(f'GM > 0.1:   {d10:,} voxels  ({100*d10/bm:.0f}% of brain mask)')

## 2. PRF R² maps

In [ ]:
fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(7 * len(SUBJECTS), 4))
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, sid in zip(axes, SUBJECTS):
    sub = Subject(sid, bids_folder=BIDS)
    t1w = sub.get_t1w()
    prf = sub.get_prf_parameters_volume(prf_deriv=PRF_DERIV)
    r2_thresh = image.math_img('img * (img >= 0.05)', img=prf['R2'])
    plotting.plot_stat_map(
        r2_thresh, bg_img=t1w, display_mode='z', cut_coords=5,
        vmin=0.05, vmax=0.3, cmap='hot', colorbar=True,
        title=f'sub-{sid}  R² (≥0.05)', axes=ax
    )
plt.tight_layout()
plt.show()

## 3. R² distribution: full brain vs V1-V3

This lets us see the noise floor in non-visual voxels vs the signal in early visual cortex.

In [ ]:
def load_prf_roi(subject, roi=None, r2_min=None):
    """Return DataFrame of PRF params, optionally masked to an ROI."""
    sub = Subject(subject, bids_folder=BIDS)
    prf_imgs = sub.get_prf_parameters_volume(prf_deriv=PRF_DERIV)

    if roi is not None:
        try:
            mask_img = sub.get_retinotopic_roi(roi, bold_space=True)
        except Exception:
            return None
    else:
        mask_img = sub.get_brain_mask(1)

    masker = NiftiMasker(mask_img=mask_img)
    data = {p: masker.fit_transform(img).squeeze() for p, img in prf_imgs.items()}
    df = pd.DataFrame(data)
    df['subject'] = subject
    df['roi'] = roi if roi else 'whole_brain'
    if r2_min is not None:
        df = df[df['R2'] >= r2_min]
    return df

fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(7 * len(SUBJECTS), 4), sharey=False)
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, sid in zip(axes, SUBJECTS):
    whole = load_prf_roi(sid)
    r2_whole = whole['R2'].values
    ax.hist(r2_whole, bins=100, range=(0, 0.3), color='grey', alpha=0.5,
            density=True, label='whole brain')
    for roi, col in zip(['V1', 'V2', 'V3'], ['C0', 'C1', 'C2']):
        df_roi = load_prf_roi(sid, roi=roi)
        if df_roi is not None and len(df_roi):
            ax.hist(df_roi['R2'].values, bins=50, range=(0, 0.3),
                    color=col, alpha=0.5, density=True, label=roi)
    ax.axvline(0.05, ls='--', c='k', lw=1, label='R²=0.05')
    ax.set_xlabel('R²')
    ax.set_ylabel('Density')
    ax.set_title(f'sub-{sid}')
    ax.legend(fontsize=8)

plt.suptitle('R² distribution: whole brain (grey) vs early visual cortex', y=1.02)
plt.tight_layout()
plt.show()

## 4. PRF size vs eccentricity per ROI

In [ ]:
import seaborn as sns

R2_THRESH = 0.05

fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(7 * len(SUBJECTS), 5), sharey=True)
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, sid in zip(axes, SUBJECTS):
    for roi, col in zip(ROI_ORDER, ROI_COLORS):
        df = load_prf_roi(sid, roi=roi, r2_min=R2_THRESH)
        if df is None or len(df) < 3:
            continue
        sns.regplot(data=df, x='ecc', y='sd', ax=ax, label=f'{roi} (n={len(df)})',
                    scatter_kws=dict(s=6, alpha=0.3, color=col),
                    line_kws=dict(lw=1.5, color=col),
                    color=col, ci=95, truncate=True)

    ax.set_xlabel('Eccentricity (°)')
    ax.set_ylabel('PRF size σ (°)')
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.set_title(f'sub-{sid}  (R²≥{R2_THRESH})')
    ax.legend(fontsize=7, ncol=2)

plt.suptitle('PRF size vs eccentricity', y=1.02)
plt.tight_layout()
plt.show()

## 5. Visual field coverage per ROI

In [ ]:
R2_THRESH = 0.05
ROIS_TO_SHOW = ['V1', 'V2', 'V3', 'hV4']

fig, axes = plt.subplots(len(SUBJECTS), len(ROIS_TO_SHOW),
                          figsize=(4 * len(ROIS_TO_SHOW), 4 * len(SUBJECTS)),
                          sharex=True, sharey=True)
axes = np.atleast_2d(axes)

fov = 3.2   # FOV radius in degrees
theta = np.linspace(0, 2 * np.pi, 100)

for i, sid in enumerate(SUBJECTS):
    for j, roi in enumerate(ROIS_TO_SHOW):
        ax = axes[i, j]
        df = load_prf_roi(sid, roi=roi, r2_min=R2_THRESH)
        if df is not None and len(df):
            ax.scatter(df['x'], df['y'], c=df['R2'], cmap='hot', s=8,
                       vmin=0.05, vmax=0.3, alpha=0.6)
        ax.add_patch(plt.Circle((0, 0), fov, fill=False, ls='--', lw=1, color='0.7'))
        ax.set_xlim(-fov, fov)
        ax.set_ylim(-fov, fov)
        ax.set_aspect('equal')
        ax.axhline(0, c='0.8', lw=0.5)
        ax.axvline(0, c='0.8', lw=0.5)
        ax.set_title(f'sub-{sid}  {roi}  (n={len(df) if df is not None else 0})', fontsize=9)
        if i == len(SUBJECTS) - 1:
            ax.set_xlabel('x (°)')
        if j == 0:
            ax.set_ylabel('y (°)')

plt.suptitle(f'Visual field coverage (R²≥{R2_THRESH})', y=1.02)
plt.tight_layout()
plt.show()

## 6. R² per ROI (box plots)

In [ ]:
fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(9 * len(SUBJECTS), 4), sharey=True)
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, sid in zip(axes, SUBJECTS):
    data_per_roi = []
    labels = []
    for roi in ROI_ORDER:
        df = load_prf_roi(sid, roi=roi)
        if df is not None and len(df):
            data_per_roi.append(df['R2'].values)
            labels.append(f'{roi}\n(n={len(df)})')

    if data_per_roi:
        bp = ax.boxplot(data_per_roi, labels=labels, patch_artist=True,
                        showfliers=False, medianprops=dict(color='k', lw=2))
        for patch, col in zip(bp['boxes'], ROI_COLORS[:len(data_per_roi)]):
            patch.set_facecolor(col)
            patch.set_alpha(0.7)

    ax.axhline(0.05, ls='--', c='k', lw=1, label='R²=0.05')
    ax.set_ylabel('R²')
    ax.set_ylim(0, 0.3)
    ax.set_title(f'sub-{sid}')
    ax.legend(fontsize=8)

plt.suptitle('R² per retinotopic ROI', y=1.02)
plt.tight_layout()
plt.show()

## 7. Median PRF parameters per ROI (table)

In [ ]:
R2_THRESH = 0.05
rows = []
for sid in SUBJECTS:
    for roi in ROI_ORDER:
        df = load_prf_roi(sid, roi=roi, r2_min=R2_THRESH)
        if df is None or len(df) < 3:
            continue
        rows.append({
            'subject': sid, 'roi': roi, 'n': len(df),
            'median_R2': df['R2'].median(),
            'median_ecc': df['ecc'].median(),
            'median_sd': df['sd'].median(),
        })

summary = pd.DataFrame(rows).set_index(['subject', 'roi'])
summary.style.format({
    'median_R2': '{:.3f}', 'median_ecc': '{:.2f}', 'median_sd': '{:.2f}'
}).background_gradient(subset=['median_R2'], cmap='Greens')